In [6]:
import pandas as pd
import numpy as np 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

'Simple script for sorting movies alphabetically'

# Reading CSV files into dataframes
movies_df = pd.read_csv("movies.csv", low_memory = False)

# Reading rating CSV file, skip bad lines since an error was returned 
ratings_df = pd.read_csv("ratings.csv", dtype={'userId': int, 'movieId': int, 'rating': float}, on_bad_lines='skip')

# Sorting the movies alphabetically, ignore special characters / case
movies_df['sort_key'] = movies_df['title'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.lower().str.strip()
movies_df = movies_df.sort_values(by = 'sort_key', ascending = True)
movies_df = movies_df.drop(columns=['sort_key']).reset_index(drop=True)

# Cleaning genre column by removing pipe separator
movies_df['genres_clean'] = movies_df['genres'].str.replace('|', ' ', regex = False)

# New column that combines title and cleaned genres
movies_df['features'] = movies_df['title'] + " " + movies_df['genres_clean']

# Creating a movies df with simpler format, to allow for compatibility with user input
movies_df['clean title'] = movies_df['title'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True).str.lower().str.strip()

tfidf = TfidfVectorizer(stop_words='english')

# Constructing TF-IDF matrix (rows = movies, columns = unique vocabulary words)
tfidf_matrix = tfidf.fit_transform(movies_df['features'])
print(f"Matrix shape: {tfidf_matrix.shape}")

# Use cosine similarity to find similarity between all movies based off of TF-IDF
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Cosine similarity matrix shape: {cosine_sim.shape}")

# Create a series mapping movie titles to index in dataframe for easier lookup
indices = pd.Series(movies_df.index, index=movies_df['title']).drop_duplicates()

def get_recommendations(title, cosine_sim = cosine_sim):
    # First check if movie is in indices, if not then return message
    if title not in indices:
        return f"Movie {title} not found in the dataset. Please check for typos or try a different movie."
    else:
        # get index of movie that matches title
        idx = indices[title]

        # get pairwise similarity scores of all movies with given movie 
        similarity_scores = list(enumerate(cosine_sim[idx]))

        # sort from most simliar to least and choose top 10
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
        similarity_scores = similarity_scores[1:11]

        # get movie indices and format as list of titles 
        movie_indices = [i[0] for i in similarity_scores]
        return movies_df['title'].iloc[movie_indices].tolist()

# test 
toystory = get_recommendations('Toy Story (1995)')
print(toystory)

# Input system for the user to enter a movie title and get recommendations
print("Movie recommendation system. Type quit or exit to leave the program at any time. \nPlease enter a movie title to get recommendations :")
while True:
    user_input = input("Movie Title :")
    if user_input.lower() in ['quit', 'exit']:
        print("Ending program.")
        break
    recommendations = get_recommendations(user_input)

    # Use isinstance to prevent errors if input is not in dataset
    if isinstance(user_input, str) and isinstance(recommendations,list):
        for i, movie in enumerate(recommendations, 1):
            print(f"  {i}. {movie}")
        print("-" * 40)
    else:
        print(recommendations)
        print("-" * 40)


Matrix shape: (9742, 9060)
Cosine similarity matrix shape: (9742, 9742)
['Toy Story 2 (1999)', 'Toy Story 3 (2010)', 'Toy, The (1982)', "We're Back! A Dinosaur's Story (1993)", 'Now and Then (1995)', 'Toy Soldiers (1991)', 'NeverEnding Story, The (1984)', 'Wild, The (2006)', 'Jumanji (1995)', 'Balto (1995)']
Movie recommendation system. Type quit or exit to leave the program at any time. 
Please enter a movie title to get recommendations :
Movie toy story  not found in the dataset. Please check for typos or try a different movie.
----------------------------------------
Ending program.
